In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time

# Reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

In [2]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze().astype("int32")
y_test = y_test.squeeze().astype("int32")

# Normalize
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Train: (50000, 32, 32, 3) (50000,)
Test : (10000, 32, 32, 3) (10000,)


In [3]:
IMG_SIZE = 96  # can use 128 if you have good GPU

def resize_images(x):
    return tf.image.resize(x, (IMG_SIZE, IMG_SIZE))

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
])

In [4]:
def build_baseline_cnn(input_shape=(32,32,3), lr=1e-3):
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [5]:
def build_transfer_model(backbone_name="MobileNetV2", lr=1e-3,
                         fine_tune=False, unfreeze_last=20):

    inputs = layers.Input(shape=(32,32,3))

    x = data_augmentation(inputs)
    x = layers.Lambda(resize_images)(x)

    # Choose backbone
    if backbone_name == "MobileNetV2":
        backbone = keras.applications.MobileNetV2(
            include_top=False, weights="imagenet",
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
        preprocess = keras.applications.mobilenet_v2.preprocess_input

    elif backbone_name == "ResNet50":
        backbone = keras.applications.ResNet50(
            include_top=False, weights="imagenet",
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
        preprocess = keras.applications.resnet.preprocess_input

    else:
        backbone = keras.applications.EfficientNetB0(
            include_top=False, weights="imagenet",
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
        preprocess = keras.applications.efficientnet.preprocess_input

    x = layers.Lambda(preprocess)(x)

    # Freeze backbone
    backbone.trainable = False

    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Fine-tuning
    if fine_tune:
        backbone.trainable = True

        for layer in backbone.layers[:-unfreeze_last]:
            layer.trainable = False

        model.compile(
            optimizer=keras.optimizers.Adam(1e-5),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

    return model

In [6]:
def train_and_eval(model, epochs=10, batch_size=64):

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )
    ]

    t0 = time.time()

    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=2
    )

    t1 = time.time()

    loss, acc = model.evaluate(x_test, y_test, verbose=0)

    print(f"Test Acc={acc*100:.2f}% | Time={t1-t0:.1f}s")

    return history, acc, (t1 - t0)

In [7]:
baseline = build_baseline_cnn()
baseline.summary()

_, acc0, time0 = train_and_eval(baseline)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 545,098 (2.08 MB)

 Trainable params: 545,098 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 - 64s - 102ms/step - accuracy: 0.4685 - loss: 1.4819 - val_accuracy: 0.5794 - val_loss: 1.2172
Epoch 2/10
625/625 - 61s - 98ms/step - accuracy: 0.6145 - loss: 1.1049 - val_accuracy: 0.6236 - val_loss: 1.0812
Epoch 3/10
625/625 - 61s - 97ms/step - accuracy: 0.6676 - loss: 0.9610 - val_accuracy: 0.6598 - val_loss: 1.0008
Epoch 4/10
625/625 - 82s - 131ms/step - accuracy: 0.7000 - loss: 0.8648 - val_accuracy: 0.6693 - val_loss: 0.9679
Epoch 5/10
625/625 - 61s - 97ms/step - accuracy: 0.7271 - loss: 0.7857 - val_accuracy: 0.6674 - val_loss: 0.9887
Epoch 6/10
625/625 - 82s - 131ms/step - accuracy: 0.7526 - loss: 0.7181 - val_accuracy: 0.6725 - val_loss: 0.9731
Epoch 7/10
625/625 - 60s - 95ms/step - accuracy: 0.7734 - loss: 0.6555 - val_accuracy: 0.6802 - val_loss: 0.9748
Test Acc=67.15% | Time=470.9s


In [ ]:
tl_model = build_transfer_model("MobileNetV2", fine_tune=False)
tl_model.summary()

_, acc1, time1 = train_and_eval(tl_model)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/10
625/625 - 321s - 514ms/step - accuracy: 0.1030 - loss: 2.4027 - val_accuracy: 0.0977 - val_loss: 2.3225
Epoch 2/10


In [ ]:
cmp_model = build_transfer_model("ResNet50", fine_tune=False)
cmp_model.summary()

_, acc3, time3 = train_and_eval(cmp_model)

In [ ]:
print("\n=== RESULTS ===")
print(f"E0 Baseline CNN       | Acc={acc0:.4f} | Time={time0:.1f}s")
print(f"E1 MobileNetV2 frozen | Acc={acc1:.4f} | Time={time1:.1f}s")
print(f"E2 Fine-tuned         | Acc={acc2:.4f} | Time={time2:.1f}s")
print(f"E3 ResNet50           | Acc={acc3:.4f} | Time={time3:.1f}s")